1. **Nome(s):** Leonardo Silveira Kiepper

2. **Matrícula(s):** 2026140417

3. **E-mail(s) institucional(is):** leonardo.kiepper@edu.ufes.br

3. **Curso(s):** Doutorado em Ciência da Computação

4. **Link do seu Colab com suas soluções (para que o professor possa acessar e rodar o código):** https://colab.research.google.com/github/LeoKiepper/Ciencia-de-Dados/blob/main/trabalho1/T1_leonardo_kiepper.ipynb


# Introdução

Cada compra que uma pessoa faz gera dados que são úteis para diversas aplicações práticas. Por exemplo, em se tratando de artigos de mercearia ou supermercado, ter um registro dos preços de produtos de interesse permite o monitoramento de gastos pessoais, precificação de receitas, etc. De modo geral, para tais aplicações serem viáveis em escala pessoal, é fundamental que a solução seja de baixo custo, de baixa manutenção, de baixo esforço e de baixa dificuldade técnica do lado do usuário. Isto elimina soluções como controle de estoque informatizado, que muitos supermercados usam.

Por exigência de lei, quando requisitado, estabelecimentos já entregam o cupom fiscal detalhando a compra. Sendo assim, o cupom fiscal é o artefato físico natural de onde extrair os dados necessários para estas aplicaçãoes, dadas as restrições acima. No entanto, a tarefa de ler cupons fiscais e extrair esses dados manualmente não é prática, devido ao volume de dados necessário para que a solução implementada tenha poder preditivo razoável.

De modo geral, os dados importantes para tais aplicações, que estariam presentes em uma nota ou cupom fiscal impresso, não são prontamente utilizáveis. Além da etapa de digitalização para converter em arquivo digital, e obrigatoriamente o reconhecimento óptico de caracteres (OCR) para converter imagem para texto, é necessário uma passada de extração de dados estruturada, na qual dados como descrição do produto e valor, são individualizados num arquivo tabular. Os dados mínimos de interesse são a descrição do produto, e o valor.

Mesmo tendo feito uma leitura e extração estruturada perfeitas, isto é, de modo que os dados constando no cupom fiscal físico tenham sido copiados exatamente para o arquivo digital, os dados ainda precisam ser curados, especificamente a descrição do produto. O desafio aqui é que a descrição do produto constando num cupom fiscal é extremamente truncada, para caber na dimensão limitada do papel térmico que normalmente é utilizado para esse propósito. Soma-se a isso os fatos de que cada comércio descreve e abrevia o produto de sua forma particular, e que, depois de identificado o produto a partir da descrição, é necessário adotar alguma forma padronizada de indexar o produto identificado para que a solução implementada seja amplamente aplicável a situações reais e possa ser generalizada para boa quantidade de casos particulares, e fica evidente que o problema não é trivial.

Limitando o escopo do projeto a predizer a descrição completa a partir da descrição truncada, procurou-se um conjunto de dados que suportasse essa tarefa. A base de dados aberta ELEVEN (https://github.com/vinidiol/descmerc) traz aproximadamente 1,1 milhão de linhas retiradas de nota fiscais anonimizadas, constando: a descrição do produto truncada, um código na Nomenclatura Comum do Mercosul (NCM) declarado pelo emissor, e um rótulo anotado por auditores fiscais especialistas. 

In [ ]:
%pip install tensorflow
%pip install itables
%pip install xgboost
%pip install torch
%pip install transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
#@title Funções auxiliares
import os, joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

DIR_ART = 'artefatos'

def _caminho(nome, ext):
    return os.path.join(DIR_ART, f'{nome}.{ext}')

def salva(nome, obj):
    """Matriz esparsa -> .npz; qualquer outro objeto -> .joblib."""
    os.makedirs(DIR_ART, exist_ok=True)
    if sp.issparse(obj):
        sp.save_npz(_caminho(nome, 'npz'), obj.tocsr())
    else:
        joblib.dump(obj, _caminho(nome, 'joblib'), compress=3)

def carrega(nome):
    if os.path.exists(_caminho(nome, 'npz')):
        return sp.load_npz(_caminho(nome, 'npz'))
    return joblib.load(_caminho(nome, 'joblib'))

def existe(nome):
    return any(os.path.exists(_caminho(nome, e)) for e in ('npz', 'joblib'))

def comoDataFrame(M, vocab, indice=None, linhas=slice(0, 20), so_nao_nulas=True):
    """Fatia de M como DataFrame, para visualização. `linhas`: slice ou lista de posições.
    Com so_nao_nulas=True, mostra só as colunas com algum valor nas linhas escolhidas."""
    sub = M[linhas]
    idx = None if indice is None else pd.Index(indice)[linhas]
    cols = np.unique(sub.indices) if so_nao_nulas else np.arange(M.shape[1])
    return pd.DataFrame(sub[:, cols].toarray(), index=idx, columns=np.asarray(vocab)[cols])

from xgboost import XGBClassifier

class XGBClassifierRotulos(XGBClassifier):
    """XGBClassifier que aceita rótulos não contíguos: no CV, classes raras podem faltar no treino do fold,
    e o XGBoost exige rótulos 0..C-1. Renumera no fit e desfaz no predict."""
    def fit(self, X, y, **kw):
        self.rotulos_, y = np.unique(y, return_inverse=True)
        return super().fit(X, y, **kw)
    def predict(self, X, **kw):
        return self.rotulos_[np.asarray(super().predict(X, **kw)).astype(int)]

import torch
from sklearn.base import BaseEstimator, ClassifierMixin

class _RedeEsparsa(torch.nn.Module):
    def __init__(self, n_entradas, larguras, dropout, layer_norm):
        super().__init__()
        self.entrada = torch.nn.EmbeddingBag(n_entradas, larguras[0], mode='sum')
        torch.nn.init.uniform_(self.entrada.weight, -n_entradas ** -0.5, n_entradas ** -0.5)
        self.vies = torch.nn.Parameter(torch.zeros(larguras[0]))
        camadas = []
        for de, para in zip(larguras[:-1], larguras[1:]):
            camadas += [torch.nn.LayerNorm(de)] if layer_norm else []
            camadas += [torch.nn.ReLU(), torch.nn.Dropout(dropout), torch.nn.Linear(de, para)]
        self.camadas = torch.nn.Sequential(*camadas)

    def forward(self, indices, deslocamentos, valores):
        return self.camadas(self.entrada(indices, deslocamentos, per_sample_weights=valores) + self.vies)

class MLPClassifierTorch(ClassifierMixin, BaseEstimator):
    def __init__(self, hidden_layer_sizes=(512,), dropout=0.2, layer_norm=False, alpha=1e-5,
                 learning_rate_init=1e-3, batch_size=512, max_iter=10, device='cuda', random_state=0):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.layer_norm = layer_norm
        self.alpha = alpha
        self.learning_rate_init = learning_rate_init
        self.batch_size = batch_size
        self.max_iter = max_iter
        self.device = device
        self.random_state = random_state

    def _lotes(self, X, tamanho):
        indices = torch.as_tensor(X.indices, dtype=torch.int64, device=self.device)
        valores = torch.as_tensor(X.data, dtype=torch.float32, device=self.device)
        for i in range(0, X.shape[0], tamanho):
            j = min(i + tamanho, X.shape[0])
            ini, fim = X.indptr[i], X.indptr[j]
            deslocamentos = torch.as_tensor(X.indptr[i:j] - ini, dtype=torch.int64, device=self.device)
            yield i, j, (indices[ini:fim], deslocamentos, valores[ini:fim])

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        sorteio = np.random.default_rng(self.random_state)
        self.classes_, y = np.unique(y, return_inverse=True)
        X = sp.csr_matrix(X, dtype=np.float32)
        rede = _RedeEsparsa(X.shape[1], [*self.hidden_layer_sizes, len(self.classes_)],
                            self.dropout, self.layer_norm).to(self.device)
        otimizador = torch.optim.AdamW(rede.parameters(), lr=self.learning_rate_init, weight_decay=self.alpha)
        agenda = torch.optim.lr_scheduler.CosineAnnealingLR(
            otimizador, self.max_iter * -(-X.shape[0] // self.batch_size))
        perda = torch.nn.CrossEntropyLoss()
        rede.train()
        for _ in range(self.max_iter):
            ordem = sorteio.permutation(X.shape[0])
            Xe, ye = X[ordem], torch.as_tensor(y[ordem], device=self.device)
            for i, j, lote in self._lotes(Xe, self.batch_size):
                otimizador.zero_grad()
                perda(rede(*lote), ye[i:j]).backward()
                otimizador.step()
                agenda.step()
        self.rede_ = rede.cpu().eval()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return self

    @torch.no_grad()
    def predict(self, X):
        rede = self.rede_.to(self.device)
        X = sp.csr_matrix(X, dtype=np.float32)
        return self.classes_[np.concatenate([rede(*lote).argmax(1).cpu().numpy()
                                             for _, _, lote in self._lotes(X, 8192)])]

import os, math, time, joblib
import numpy as np
import torch
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoModel, AutoTokenizer

_NAO_AFETAM_TREINO = ('verbose', 'lote_inferencia', 'device', 'arq_checkpoint')


class _CyclicLRKtrain:
    """ktrain.lroptimize.triangular.CyclicLR como usado por fit_onecycle: lr sobe de lr/10 a lr na 1a metade e desce
    até lr/1000 na 2a; beta1 faz o caminho inverso (0.95 -> 0.85 -> 0.95). Atualiza ao fim de cada lote."""
    def __init__(self, lr, n_passos, max_momentum=0.95, min_momentum=0.85):
        self.max_lr, self.base_lr, self.orig_base_lr = lr, lr / 10, lr / 10
        self.step_size = math.ceil(n_passos / 2)
        self.max_m, self.min_m = max_momentum, min_momentum
        self.it, self.overhump = 0, False

    def apos_lote(self):
        self.it += 1
        ciclo = math.floor(1 + self.it / (2 * self.step_size))
        x = abs(self.it / self.step_size - 2 * ciclo + 1)
        lr = self.base_lr + (self.max_lr - self.base_lr) * max(0.0, 1 - x)
        anterior, its = self.overhump, (self.it + 1) % (self.step_size * 2)
        self.overhump = its / self.step_size > 1
        if not anterior and self.overhump:
            self.base_lr = self.max_lr / 1000
        elif anterior and not self.overhump:
            self.base_lr = self.orig_base_lr
        pct = 1.0 - (its - self.step_size) / self.step_size if self.overhump else its / self.step_size
        return lr, self.max_m - pct * (self.max_m - self.min_m)


class _RedeBERT(torch.nn.Module):
    def __init__(self, config, n_classes, checkpoint=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained(checkpoint) if checkpoint else AutoModel.from_config(config)
        self.saida = torch.nn.Linear(config.hidden_size, n_classes)
        torch.nn.init.xavier_uniform_(self.saida.weight)
        torch.nn.init.zeros_(self.saida.bias)

    def forward(self, ids):
        return self.saida(self.bert(input_ids=ids, attention_mask=(ids != 0).long()).pooler_output)


class ClassificadorBERT(BaseEstimator, ClassifierMixin):
    """Estimador sklearn sobre o texto tokenizado (X_texto: tokens do preprocessamento unidos por espaço), convertido
    em ids do mBERT: [CLS] ... [SEP], padding 0 até maxlen, minúsculas e sem acentos (Tokenizer do keras_bert,
    cased=False). predict devolve códigos 0..n_classes-1; n_classes=None usa max(y)+1.
    corta_padding: cada lote é cortado no maior texto dele; a máscara de atenção já ignora o padding, então o resultado
    é o mesmo de usar maxlen inteiro, só mais rápido. arq_checkpoint: estado ao fim de cada época, para retomar o fit;
    apagado quando o fit termina."""
    def __init__(self, checkpoint='google-bert/bert-base-multilingual-cased', maxlen=150, batch_size=8, lr=2e-5,
                 epocas=3, validacao=0.1, n_classes=None, corta_padding=True, semente=0,
                 lote_inferencia=512, arq_checkpoint=None, device='cuda', verbose=1):
        self.checkpoint, self.maxlen, self.batch_size, self.lr = checkpoint, maxlen, batch_size, lr
        self.epocas, self.validacao, self.n_classes = epocas, validacao, n_classes
        self.corta_padding, self.semente, self.lote_inferencia = corta_padding, semente, lote_inferencia
        self.arq_checkpoint, self.device, self.verbose = arq_checkpoint, device, verbose

    def _dev(self):
        return self.device if torch.cuda.is_available() or self.device == 'cpu' else 'cpu'

    def _ids(self, X):
        if getattr(self, 'tokenizador_', None) is None:
            self.tokenizador_ = AutoTokenizer.from_pretrained(self.checkpoint, do_lower_case=True)
        ids = self.tokenizador_([str(s) for s in X], padding='max_length', truncation=True, max_length=self.maxlen)['input_ids']
        return np.asarray(ids, dtype=np.int32)

    def _lote(self, ids, idx):
        ids = ids[idx]
        if self.corta_padding:
            ids = ids[:, :int((ids != 0).sum(1).max())]
        return ids

    def _logits(self, ids, lote):
        self.modelo_.eval()
        saidas = []
        with torch.no_grad():
            for i in range(0, len(ids), lote):
                saidas.append(self.modelo_(self._lote(ids, slice(i, i + lote))).float())
        return torch.cat(saidas)

    def _log(self, *a):
        if self.verbose:
            print(*a, flush=True)

    def fit(self, X, y):
        torch.set_float32_matmul_precision('high')
        torch.manual_seed(self.semente)
        dev = self._dev()
        ids, y = self._ids(X), np.asarray(y)
        n_classes = self.n_classes or int(y.max()) + 1
        i_tr, i_val = (train_test_split(np.arange(len(y)), test_size=self.validacao, random_state=self.semente)
                       if self.validacao else (np.arange(len(y)), None))
        idg = torch.as_tensor(ids, dtype=torch.long, device=dev)
        yg = torch.as_tensor(y, dtype=torch.long, device=dev)

        self.config_ = AutoConfig.from_pretrained(self.checkpoint)
        self.modelo_ = _RedeBERT(self.config_, n_classes, self.checkpoint).to(dev)
        otim = torch.optim.Adam(self.modelo_.parameters(), lr=self.lr / 10, betas=(0.9, 0.999), eps=1e-7)
        passos_epoca = math.ceil(len(i_tr) / self.batch_size)
        agenda = _CyclicLRKtrain(self.lr, passos_epoca * self.epocas)
        perda = torch.nn.CrossEntropyLoss()
        self.historico_, ep0 = [], 0

        chave = joblib.hash(({k: v for k, v in self.get_params().items() if k not in _NAO_AFETAM_TREINO}, ids, y))
        ck = self.arq_checkpoint
        if ck and os.path.exists(ck):
            estado = torch.load(ck, map_location=dev, weights_only=False)
            if estado['chave'] == chave:
                self.modelo_.load_state_dict(estado['modelo']); otim.load_state_dict(estado['otim'])
                agenda.__dict__.update(estado['agenda']); self.historico_, ep0 = estado['historico'], estado['epoca']
                self._log(f'retomando do checkpoint: {ep0}/{self.epocas} épocas concluídas')

        for ep in range(ep0, self.epocas):
            self.modelo_.train()
            ordem = np.random.default_rng(self.semente + ep).permutation(i_tr)
            t, soma, n = time.perf_counter(), 0.0, 0
            for k in range(passos_epoca):
                idx = torch.as_tensor(ordem[k * self.batch_size:(k + 1) * self.batch_size], device=dev)
                l = perda(self.modelo_(self._lote(idg, idx)), yg[idx])
                otim.zero_grad(set_to_none=True); l.backward(); otim.step()
                lr, beta1 = agenda.apos_lote()
                for g in otim.param_groups:
                    g['lr'], g['betas'] = lr, (beta1, g['betas'][1])
                if k % 100 == 0:
                    soma, n = soma + l.item(), n + 1
                if k % max(1, passos_epoca // 20) == 0:
                    dt = time.perf_counter() - t
                    self._log(f'época {ep + 1}/{self.epocas} passo {k}/{passos_epoca} perda {soma / max(n, 1):.4f} '
                              f'lr {lr:.2e} | {dt / 60:.1f} min, faltam ~{dt / (k + 1) * (passos_epoca - k - 1) / 60:.0f} min')
            reg = {'epoca': ep + 1, 'perda_treino': soma / max(n, 1), 'tempo_s': time.perf_counter() - t}
            if i_val is not None:
                iv = torch.as_tensor(i_val, device=dev)
                lg, yv = self._logits(idg[iv], self.lote_inferencia), yg[iv]
                reg.update(perda_val=float(perda(lg, yv)), acuracia_val=float((lg.argmax(1) == yv).float().mean()),
                           f1_macro_val=float(f1_score(yv.cpu().numpy(), lg.argmax(1).cpu().numpy(), average='macro')))
            self.historico_.append(reg)
            self._log({k: round(v, 4) if isinstance(v, float) else v for k, v in reg.items()})
            if ck:
                torch.save({'chave': chave, 'epoca': ep + 1, 'modelo': self.modelo_.state_dict(), 'otim': otim.state_dict(),
                            'agenda': dict(agenda.__dict__), 'historico': self.historico_}, ck)

        self.indice_validacao_ = i_val
        self.score_validacao_ = self.historico_[-1].get('f1_macro_val', float('nan'))
        if ck and os.path.exists(ck):
            os.remove(ck)
        del idg, yg, otim
        return self

    def predict(self, X):
        self._garante_modelo()
        ids = torch.as_tensor(self._ids(X), dtype=torch.long, device=self._dev())
        return self._logits(ids, self.lote_inferencia).argmax(1).cpu().numpy()

    def _garante_modelo(self):
        if getattr(self, 'modelo_', None) is None:
            self.modelo_ = _RedeBERT(self.config_, self.n_saidas_).to(self._dev())
            self.modelo_.load_state_dict(self.pesos_)
            del self.pesos_

    def __getstate__(self):
        estado = {k: v for k, v in self.__dict__.items() if k != 'modelo_'}
        if getattr(self, 'modelo_', None) is not None:
            estado['pesos_'] = {k: v.detach().cpu() for k, v in self.modelo_.state_dict().items()}
            estado['n_saidas_'] = self.modelo_.saida.out_features
        return estado

    def __setstate__(self, estado):
        self.__dict__.update(estado)
        self.modelo_ = None


In [1]:
#@title Carrega o dataset

import os
import pandas as pd
from langdetect import detect, DetectorFactory, LangDetectException
from itables import show
# Carrega a partir do github
base = pd.read_csv('https://github.com/LeoKiepper/Ciencia-de-Dados/raw/main/trabalho2/descmerc/BaseDESC_NCM.zip',
                   sep='\t', dtype={'XPROD': str, 'NCM': str})

show(base)
show(base['NCM'].value_counts())
print("Quantas classes resultam em cada nível do NCM")
print({k: base['NCM'].str[:k].nunique() for k in (2, 4, 6, 8)})


Quantas classes resultam em cada nível do NCM
{2: 82, 4: 703, 6: 1628, 8: 2174}


# Metodogia

### Pré-processamento

Os dados da coluna `NCM` são tratados como labels, e passados por um *label encoder*. A coluna `XPROD` tem dados formatados como *string*. Para tratar a imprevisibilidade do estilo de escrita de cada descrição, foi definida a função de preprocessamento abaixo.

A lógica por trás dessa função é remover ruído das descrições que não contribuiriam para descrever o produto, tomando cuidado para não remover sinal junto com o ruído, levando em conta características específicas de como pessoas escrevem essas descrições. Duas regras globais nortearam a implementação do pré-processamento:

>"Qualificadores que sejam aplicáveis a múltiplos produtos de múltiplas categorias não contribuem para individualizar o produto para uma categoria, então são ruído". (Por exemplo, características da embalagem, marketing como 'leve X pague Y', quantidades)

>"Se há risco de eliminar sinal, não remover um qualificador. A informação passa a ser dado para o algoritimo aprender a usar ou ignorar.".

Um caso notável de qualificadores que foram mantidos pela segunda regra são cores. Para alguns produtos, a cor é um qualificador que não muda a função. Por exemplo, uma escova de dente pode ser de qualquer cor, e ter exatamente a mesma função. Para outros, a cor é um qualificador que distingue o produto de outros semelhantes. Por exemplo, vinho branco é diferente de vinho tinto.

Com base nessas duas regras, foi criada uma sequência de regex para filtrar as strings. Esses regex removem grafias observadas para quantidades (que em geral não seguem grafia padronizada de unidades de medida, como 'metro' → 'm', 'litro' → 'l', 'grama' → 'g, etc). Depois de filtradas, as strings são tokenizadas. Alguns problemas são atacados em mais de uma etapa, para retirá-los de formas diferentes que não são fáceis de pegar por uma regra única. São feitas as operações listadas abaixo. Para listagens completas das chaves regex usadas, vide o código da célula seguinte.

1. Remover acentos e quantidades/tamanhos de embalagem
2. Dividir o texto em tokens
3. Converter o texto para letras minúsculas
4. Separar tokens que usam '.' ou '/' como separador
5. Remover símbolos de pontuação de cada token e descartar tokens vazios
6. Separar palavra colada a uma quantidade (fugini100gr -> fugini, 100gr)
7. Remover unidades, embalagens, marketing, tamanho, referência e stop words
8. Remover letras soltas (tokens com apenas uma letra)
9. Remover números soltos (3, 1/2, 030003, 20x25) e quantidades (750ml, 1kg)
10. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)




In [3]:
#@title Tokeniza o dataset
import re, unicodedata
import pandas as pd
import nltk
import string
from nltk import word_tokenize
from nltk.corpus import stopwords
from itables import show
from collections import Counter
from functools import lru_cache

nltk.download('punkt_tab')
nltk.download('stopwords')

NUM  = r'\d+(?:[.,]\d+)?'
UNID = r'(?:M2|MLN|ML|MLS|M|MT|MTS|G|GR|GRS|KG|KGS|L|LT|LTS|LI|LITROS?|UN|UND|UNDS|UNID|U|PC|PCS|KIT)'
EMB  = r'(?:CX|CXA|UN|UND|PT|PC|FD|DP|SC|CJ|PCT|BD|FR|GA|GF|DZ)'
FIM  = r'\.?(?![A-Z\d])'          # fim do token: não seguido de letra ou dígito
PADRAO_QTD = re.compile(rf'''
    (?<![\w,.]){EMB}?(?:{NUM}\s?X\s?)+{NUM}\s?{UNID}{FIM}   # 12X500ML, DP12X24X22G
  | (?<![\w,.]){EMB}(?:{NUM}\s?X\s?)+{NUM}{FIM}             # CX20X50
  | (?<![\w,.])1\s?X\s?\d+{FIM}                             # 1 X 12
  | (?<![\w,.]){EMB}?{NUM}\s?{UNID}{FIM}                    # 500ML, 1,5 L, CX10KG
  | \b{EMB}\s?[/-]\s?\d+(?:\s?[/-]\s?{EMB})?{FIM}           # CX/12, UN/0001, CX-6-UN
  | \b{EMB}\d+{EMB}?{FIM}                                   # PT0001UN
  | \bC/\s?\d+(?:\s?{UNID})?{FIM}                           # C/12, C/20MTS
  | \bQTD\.?\s?{NUM}(?:\s{EMB})?                            # QTD. 2.00 DP
  | \(\s?\d+\s?{UNID}?\s?\)                                 # (12 UN)
  | \(CX[\d,]+(?:M2|PCS?)\)                                 # (CX1,62M2)
  | -\s?\d+VL\b                                             # - 24VL
  | \bF\.?P\.?S\.?\s?-?\s?\d+\+?                            # FPS 70
  | \bLV?\d+\s?PG?\d+(?:ML?|G|GR)?\b                        # LV12PG11, L4P3, LV16 PG15
  | \b\d+\s?{EMB}\s?\d+\s?{UNID}{FIM}                       # 1CX12UN
''', re.X)
ALFANUM = re.compile(r'(?=.*\d)(?=.*[a-z])[a-z\d]+')    # tem letra e dígito
FRAG    = re.compile(r'[a-z]{3,}')                      # fragmento alfabético resgatável

# '.' e '/' usados como separador (não quebra decimais e frações: 2.5mm, 1/2)
SEP = re.compile(r'(?<!\d)[./]|[./](?!\d)')
TABELA_PONT = str.maketrans('', '', string.punctuation)
# número+unidade dentro de token (CAB.SAV.750ML -> 750ml, 1-KG -> 1kg, 12x40g)
QTD_TOKEN = re.compile(r'(?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs)')
# palavra colada a uma quantidade (FUGINI100GR -> fugini + 100gr; HARIBO12X350G -> haribo + 12x350g)
COLADO = re.compile(r'([a-z]{2,})((?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs))')

# tokens só de dígitos, separadores e 'x': 3, 1/2, 030003, 20x25
NUMERICO = re.compile(r'[\d.,/+\-%*:;"\'&#x]*\d[\d.,/+\-%*:;"\'&#x]*')
# códigos de referência (preserva DN15, RJ11, IP65, E27, MM2, CA50 e medidas como 2700K, 6800W)
CODIGO = re.compile(r'(?:ref|cod|mva|lote|venc|nr|n\.)[.:\-]?\w[\w.\-/]*'          # REF 123, MVA:29
                    r'|[a-z]-?\d+[a-z]?'                                          # C21, R0091, S08
                    r'|[a-z]-\d+/\d+'                                             # B-4472/17
                    r'|\d{4,}-?(?!(?:k|w|v|a|kv|ka|va|mm|cm|m)$)[a-z]{1,2}'        # 27814E, 1328-M
                    r'|[a-z]{1,2}\d{4,}'                                          # MK2431
                    r'|r\.\d+|a/\d+|tam\d+|\d+h\d+'                               # R.123, A/4552, TAM2, 1H15
                    r'|(?!dn|rj|ip|mm|ca|e\d)[a-z]{2}-?\d+[a-z]?')                # SG-8407, GL83, BR-01
# dimensões NxM(xP), com ou sem unidade: 20x25, 67x370mm, 32mmx22mm, 2750x1840x15mm, 19mmx10m
DIMENSAO = re.compile(r'(?:\d+(?:mm|cm|mts?|m)?x)+\d+(?:mm|cm|mts?|m)?')

def _sem_acento(s): return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

RUIDO = (
    {'ml','mls','gr','grs','kg','kgs','lt','lts','un','und','unid','cx','cxa','fd','dp','pct','pc','pcs',
     'mt','mts','pt','fr','sc','bd','gf'}                                         # unidades e embalagens
  | {'pack','sache','refil','cartela','pet','lata'}                               # embalagem
  | {'premium','prem','especial','classic','classico','class','super','ultra','plus','max','maxi','top',
     'gold','golden','star','prime','master','power','smart','easy','life','ideal','royal','imperial',
     'new','novo','nova','tradic','orig','duo','one'}                             # marketing / linha
  | {'grande','gde','pequeno','pequena','peq'}                                    # tamanho
  | {'ref','cod','nr'}                                                            # referência
  | ({_sem_acento(w) for w in stopwords.words('portuguese')} - {'com','sem','nao'})
)

@lru_cache(maxsize=None)
def _tokeniza(df: pd.Series):
    # 1. Remover acentos e quantidades/tamanhos de embalagem (sobre a string)
    texto = PADRAO_QTD.sub(' ', _sem_acento(df).upper())

    # 2. Dividir o texto em tokens
    tokens = word_tokenize(texto)

    # 3. Converter o texto para letras minúsculas
    tokens = [w.lower() for w in tokens]

    # 4. Separar tokens que usam '.' ou '/' como separador (VIN.PORT.CHAMINE, P/TRAT)
    tokens = [p for w in tokens for p in SEP.split(w)]

    # 5. Remover símbolos de pontuação de cada token e descartar tokens vazios
    tokens = [w.translate(TABELA_PONT) for w in tokens]
    tokens = [w for w in tokens if w]

    # 6. Separar palavra colada a uma quantidade (fugini100gr -> fugini, 100gr)
    tokens = [p for w in tokens for m in [COLADO.fullmatch(w)] for p in (m.groups() if m else (w,))]

    # 7. Remover unidades, embalagens, marketing, tamanho, referência e stop words
    tokens = [w for w in tokens if w not in RUIDO]

    # 8. Remover letras soltas
    tokens = [w for w in tokens if not (len(w) == 1 and w.isalpha())]

    # 9. Remover números soltos (3, 1/2, 030003), quantidades (750ml, 1kg) e dimensões (67x370mm)
    tokens = [w for w in tokens if not NUMERICO.fullmatch(w)
                               and not QTD_TOKEN.fullmatch(w)
                               and not DIMENSAO.fullmatch(w)]

    # 10. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)
    tokens = [w for w in tokens if not CODIGO.fullmatch(w)]

    return tokens
  
# Em quantas descrições da base cada token aparece (calculado uma vez)
DF_TOKENS = Counter(t for toks in base['XPROD'].map(_tokeniza) for t in set(toks))

def resgata(tokens):
    """Alfanuméricos que aparecem em no máximo uma descrição da base são trocados
    pelos seus fragmentos alfabéticos (>= 3 letras, fora do RUIDO); sem fragmento, somem."""
    out = []
    for w in tokens:
        if DF_TOKENS[w] <= 1 and ALFANUM.fullmatch(w):
            out += [p for p in FRAG.findall(w) if p not in RUIDO]
        else:
            out.append(w)
    return out

def preprocessamento(df: pd.Series):
    # Tokeniza e resgata fragmentos de alfanuméricos únicos
    return resgata(_tokeniza(df))

[nltk_data] Error loading punkt_tab: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.
[nltk_data] Error loading stopwords: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to 

Os textos tokenizados são passados para um vetorizador TF-IDF para conformar os dados para os algoritimos investigados, com uma exceção: para atender a requisitos técnicos de formatação de dados do algoritimo BERT, os textos tokenizados são reformatados como string, aproveitando o fato de a operação de tokenização preservar a sequência dos tokens. Isso é feito dessa forma para que todos os algoritimos investigados utilizem os mesmos dados, dessa forma retirando uma variável interveniente da avaliação que foi efetuada


In [4]:
#@title Calcula a matriz TF-IDF do dataset completo
import os, io, urllib.request
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

REPO_RAW = 'https://raw.githubusercontent.com/LeoKiepper/Ciencia-de-Dados/main/trabalho2/'
ARQ_X = _caminho('X_tfidf', 'npz')
ARQ_VOCAB = _caminho('X_tfidf_vocab', 'csv')

def calculaX(x: pd.Series):
    """TF-IDF de x. Devolve (M, vocab): M é CSR float32 (linhas na ordem de x), vocab é o array de tokens."""
    vectorizer = TfidfVectorizer(analyzer=preprocessamento, dtype=np.float32)
    M = vectorizer.fit_transform(x)
    return M, vectorizer.get_feature_names_out()

def _le_vocab(origem):
    return pd.read_csv(origem, keep_default_na=False)['token'].to_numpy()

def _baixa_do_github():
    M = sp.load_npz(io.BytesIO(urllib.request.urlopen(REPO_RAW + ARQ_X).read()))
    return M, _le_vocab(REPO_RAW + ARQ_VOCAB)

def _grava(M, vocab):
    os.makedirs(DIR_ART, exist_ok=True)
    sp.save_npz(ARQ_X, M)
    pd.Series(vocab, name='token').to_csv(ARQ_VOCAB, index=False)

def carregaX(x: pd.Series):
    if os.path.exists(ARQ_X) and os.path.exists(ARQ_VOCAB):
        return sp.load_npz(ARQ_X), _le_vocab(ARQ_VOCAB)
    try:
        M, vocab = _baixa_do_github()
    except Exception:
        M, vocab = calculaX(x)
    _grava(M, vocab)
    return M, vocab


In [ ]:
#@title Recompõe o texto tokenizado do dataset completo
import os
import numpy as np
import pandas as pd

ARQ_TEXTO = _caminho('X_texto', 'csv.gz')

def calculaTexto(x: pd.Series):
    """Tokens de preprocessamento(x) unidos por espaço, na ordem em que aparecem (linhas na ordem de x).
    Nenhum token contém espaço, então a lista de tokens é recuperável com split(' ')."""
    return x.map(preprocessamento).str.join(' ').to_numpy()

def _le_texto(origem):
    return pd.read_csv(origem, dtype={'texto': str}, keep_default_na=False)['texto'].to_numpy()

def _grava_texto(T):
    os.makedirs(DIR_ART, exist_ok=True)
    pd.Series(T, name='texto').to_csv(ARQ_TEXTO, index=False)

def carregaTexto(x: pd.Series):
    if os.path.exists(ARQ_TEXTO):
        return _le_texto(ARQ_TEXTO)
    try:
        T = _le_texto(REPO_RAW + ARQ_TEXTO)
    except Exception:
        T = calculaTexto(x)
    _grava_texto(T)
    return T


### Definição de variáveis de entrada, hiperparâmetros e indicadores de desempenho

O trabalho-referência faz predição sobre a coluna `rotulo` a partir de `XPROD`. Por esse motivo, os resultados trazidos lá não serão diretamente comparáveis com o presente trabalho. Para obter um termo de comparação, o algoritimo BERT, que lá obteve o melhor resultado, foi aplicado para a predição do `NCM` a partir de `XPROD`. Além disso, foi testado *Random Forest* (RF) como baseline para detecção de sinal, maquina de vetor de suporte (SVM), que até hoje é um algoritimo importante, XGBoost, que está sendo muito usado atualmente na literatura, e *multi-layer perceptron*.

Para medir o desempenho das predições, foi selecionado o *F1-score* como objetivo de otimização, pois, nessa aplicação, o *dataset* é desbalanceado. Por isso, otimizar para acurácia poderia levar a modelos enganosamente "bons", com capacidade preditiva boa para a classes bem representadas, mas pobre em classes raras. Mesmo assim, para obter uma contextualização mais completa dos resultados dos demais algoritimos contra o BERT e contra o trabalho-referência, é mostrado também a acurácia, pois esse foi o indicador de desempenho escolhido lá.

Os algoritimos de classificação testados devem exprimir relações entre a descrição do produto (coluna `XPROD` do *dataset*, entrada) e o código `NCM` (saída, ou predição), até os 4 primeiros dígitos. O *dataset* completo é passado por um split, reservando 25% para teste e 75% para treino, depois são aplicados os respectivos pré-processamentos. Os modelos foram treinados para uma mesma semente de split, do *dataset* completo.

Cada algoritimo de classificação foi treinado com validação cruzada usando a biblioteca `GridSearchCV`. O grid de hiperparâmetros de cada um dos algoritimos é explicado abaixo;


In [ ]:
#@title Define variáveis de entrada para os algoritimos de classificação
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


USAR_AMOSTRA = False
N_AMOSTRA = 1000
TEST_SIZE = 0.25
NIVEL_NCM = 4
SEMENTE_AMOSTRA = None
SEMENTES_SPLIT = [345510517, 1909508899, 1596260650, 1363945693, 1375679088, 827915821, 1257687752, 686578392, 378699650, 259128988]

# Gerar splits localmente. Rodar as células:
# 1. Funções auxiliares
# 2. Carrega o dataset
# 3. Tokeniza o dataset
# 4. Calcula a matriz TF-IDF
# 5. Recompõe o texto tokenizado
# 6. Define variáveis… (gera os splits)
# 7. Suite de treino
# 8. A célula de treino da máquina: LCAD, Notebook pessoal ou DGX

if USAR_AMOSTRA:
    semente_amostra = SEMENTE_AMOSTRA if SEMENTE_AMOSTRA is not None else int(np.random.randint(0, 2**31))
    df = base.sample(n=N_AMOSTRA, random_state=semente_amostra)
else:
    semente_amostra = None
    df = base

# linhas de X_full na ordem de df
X_full, vocab = carregaX(base['XPROD'])
X = X_full[base.index.get_indexer(df.index)]
X_texto = carregaTexto(base['XPROD'])[base.index.get_indexer(df.index)]
le = LabelEncoder()
y = pd.Series(le.fit_transform(df['NCM'].str[:NIVEL_NCM]), index=df.index, name='NCM')

print(f'X {X.shape} {X.dtype}, {X.nnz} valores não nulos')
show(comoDataFrame(X, vocab, indice=df.index))
print(pd.Series(vocab).sample(50).tolist())


# Separa uma fatia do dataset para teste
for semente in SEMENTES_SPLIT:
    X_train, X_test, X_train_texto, X_test_texto, y_train, y_test = train_test_split(
        X, X_texto, y, test_size=TEST_SIZE, random_state=semente)
    for nome, obj in {'X_train': X_train, 'X_test': X_test, 'X_train_texto': X_train_texto,
                      'X_test_texto': X_test_texto, 'y_train': y_train, 'y_test': y_test}.items():
        salva(f'{nome}_splitseed{semente}', obj)
salva('le', le)

salva('config_dados', {'usar_amostra': USAR_AMOSTRA,
                       'n_amostra': len(df),
                       'semente_amostra': semente_amostra,
                       'test_size': TEST_SIZE,
                       'sementes_split': SEMENTES_SPLIT,
                       'guardou_matrizes': not (USAR_AMOSTRA and SEMENTE_AMOSTRA is None),
                       'nivel_ncm': NIVEL_NCM
})

NameError: name 'base' is not defined

In [ ]:
#@title Suite de treino
import os, glob, json, time, datetime, joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, ParameterGrid

IGNORA_HASH = {'n_jobs', 'verbose'}

def f1_macro_em_lotes(estimador, X, y, lote=2000):
    yhat = np.concatenate([estimador.predict(X[i:i + lote]) for i in range(0, X.shape[0], lote)])
    return f1_score(y, yhat, average='macro')

def _caminho_descricao(pasta):
    return os.path.join(DIR_ART, pasta, 'descricao.json')

def _le_descricao(pasta):
    with open(_caminho_descricao(pasta)) as f:
        return json.load(f)

def _grava_descricao(pasta, descricao):
    with open(_caminho_descricao(pasta), 'w') as f:
        json.dump(descricao, f, indent=2, ensure_ascii=False, default=repr)

def _pasta_com_hash(h):
    for arq in sorted(glob.glob(os.path.join(DIR_ART, 'grid_*', 'descricao.json'))):
        pasta = os.path.basename(os.path.dirname(arq))
        if _le_descricao(pasta)['hash'] == h:
            return pasta
    return None

PREFIXO_SPLIT = 'y_train_splitseed'

def sementes_split():
    return sorted(int(a[len(PREFIXO_SPLIT):-len('.joblib')]) for a in os.listdir(DIR_ART)
                  if a.startswith(PREFIXO_SPLIT) and a.endswith('.joblib'))

def grid_otimizado(nome):
    return {k: [v] for k, v in carrega(f'treino_{nome}')['melhores_parametros'].items()}

def _descreve(estimador, grids, linhas_refit, semente, cv=5):
    desc = {'estimador': f'{type(estimador).__module__}.{type(estimador).__name__}',
            'parametros': {k: v for k, v in estimador.get_params(deep=False).items() if k not in IGNORA_HASH},
            'grid': grids,
            'semente_split': semente}
    if linhas_refit is not None:
        desc['linhas_refit'] = linhas_refit
    if cv is None:
        desc['cv'] = None
    return desc

def _ja_treinado(rotulo, pasta, h):
    return (_le_descricao(pasta)['concluido'] and existe(f'modelo_{rotulo}') and existe(f'treino_{rotulo}')
            and carrega(f'treino_{rotulo}').get('hash') == h)

def _nova_pasta(rotulo, h, desc):
    agora = datetime.datetime.now()
    pasta = f'grid_{agora:%Y%m%d_%H%M%S}_{rotulo}'
    os.makedirs(os.path.join(DIR_ART, pasta))
    _grava_descricao(pasta, {'nome': rotulo, 'hash': h, **desc,
                             'criada': f'{agora:%Y-%m-%d %H:%M:%S}', 'concluido': False})
    return pasta
def _busca_por_partes(nome, estimador, grids, pasta, X_train, y_train, cv, scoring, n_jobs, verbose):
    partes = []
    for i, g in enumerate(grids, 1):
        arq = f'{pasta}/parte_{i:03d}'
        if existe(arq):
            partes.append(carrega(arq))
            print(f'[{nome} {i}/{len(grids)}] já salva, pulando')
            continue
        busca = GridSearchCV(clone(estimador), g, cv=cv, scoring=scoring,
                             n_jobs=n_jobs, refit=False, verbose=verbose)
        t = time.perf_counter()
        busca.fit(X_train, y_train)
        parte = {'grid': g, 'cv_results': pd.DataFrame(busca.cv_results_), 'tempo_s': time.perf_counter() - t}
        salva(arq, parte)
        partes.append(parte)
        print(f'[{nome} {i}/{len(grids)}] {parte["tempo_s"] / 60:.1f} min; '
              f'melhor {parte["cv_results"].mean_test_score.max():.4f}')
    return partes

def _junta_partes(partes):
    cv_results = pd.concat([p['cv_results'] for p in partes], ignore_index=True)
    cv_results['rank_test_score'] = cv_results.mean_test_score.rank(ascending=False, method='min', na_option='bottom').astype(int)
    return cv_results


def _ponto_unico(grids):
    """Sem CV (cv=None): o grid deve ter um único ponto, que é o escolhido por definição."""
    pontos = list(ParameterGrid(grids))
    assert len(pontos) == 1, f'sem CV o grid deve ter um único ponto; tem {len(pontos)}'
    return {'params': pontos[0], 'mean_test_score': float('nan')}

def _amostra_para_refit(X, y, linhas):
    if linhas is None or linhas >= len(y):
        return X, y
    posicoes = np.sort(np.random.default_rng(0).choice(len(y), linhas, replace=False))
    return X[posicoes], y.iloc[posicoes]

def _treina(nome, estimador, grid, X_train, y_train, semente, cv=5, scoring=f1_macro_em_lotes,
            n_jobs=1, verbose=3, refaz=False, linhas_refit=None):
    grids = grid if isinstance(grid, list) else [grid]
    if len(ParameterGrid(grids)) == 1:
        cv = None
    rotulo = f'{nome}_splitseed{semente}'
    desc = _descreve(estimador, grids, linhas_refit, semente, cv)
    h = joblib.hash(desc)

    pasta = None if refaz else _pasta_com_hash(h)
    if pasta is not None and _ja_treinado(rotulo, pasta, h):
        print(f'[{rotulo}] já treinado com este grid ({pasta}, hash {h[:8]}), pulando')
        return
    pasta = pasta or _nova_pasta(rotulo, h, desc)
    if cv is None:
        print(f'[{rotulo}] {DIR_ART}/{pasta}/ (hash {h[:8]}): grid de ponto único, treino direto sem CV')
        partes, cv_results, melhor = [], None, _ponto_unico(grids)
    else:
        print(f'[{rotulo}] partes em {DIR_ART}/{pasta}/ (hash {h[:8]}): {len(grids)} partes × {cv} folds')
        partes = _busca_por_partes(rotulo, estimador, grids, pasta, X_train, y_train, cv, scoring, n_jobs, verbose)
        cv_results = _junta_partes(partes)
        melhor = cv_results.loc[cv_results.mean_test_score.idxmax()]

    X_refit, y_refit = _amostra_para_refit(X_train, y_train, linhas_refit)
    t = time.perf_counter()
    modelo = clone(estimador).set_params(**melhor['params']).fit(X_refit, y_refit)
    t_refit = time.perf_counter() - t
    tempo = sum(p['tempo_s'] for p in partes) + t_refit

    salva(f'modelo_{rotulo}', modelo)
    salva(f'treino_{rotulo}', {
        'melhores_parametros': melhor['params'],
        'parametros': modelo.get_params(),
        'melhor_score_cv': melhor['mean_test_score'],
        'cv_results': cv_results,
        'tempo_s': tempo,
        'indice_treino': y_train.index,
        'indice_refit': y_refit.index,
        'pasta_partes': pasta,
        'hash': h,
        'semente_split': semente,
    })
    _grava_descricao(pasta, {**_le_descricao(pasta), 'concluido': True, 'tempo_s': tempo,
                             'melhor_score_cv': float(melhor['mean_test_score']),
                             'melhores_parametros': melhor['params']})
    print(f'[{rotulo}] treino {t_refit / 60:.1f} min; parâmetros: {melhor["params"]}')
    del modelo

def _opcoes_do_par(extras):
    return extras[0] if extras else {}

def treina(modelos, entrada, **opcoes):
    sementes = sementes_split()
    assert sementes, f'nenhum {PREFIXO_SPLIT}* em {DIR_ART}'
    for semente in sementes:
        sufixo = f'_splitseed{semente}'
        X_train, y_train = carrega(f'{entrada}{sufixo}'), carrega(f'y_train{sufixo}')
        for nome, estimador, grid, *extras in modelos:
            _treina(nome, estimador, grid, X_train, y_train, semente, **{**opcoes, **_opcoes_do_par(extras)})
        del X_train, y_train

In [ ]:
#@title Treina os algoritmos (Notebook LCAD)
import numpy as np
from sklearn.ensemble import RandomForestClassifier


RF_PRODUTO_TL = 600_000
RF_L = np.geomspace(4000, 150000, 16).round(-1).astype(int)
RF_PARES = sorted({(int(max(1, RF_PRODUTO_TL // L)), int(L)) for L in RF_L})
RF_BASE = {'max_features': ['sqrt'], 'class_weight': [None, 'balanced_subsample'], 'bootstrap': [True, False]}

XGB_PARES = [(150, 0.3), (300, 0.15), (450, 0.1)]
XGB_COMBINACOES_TESTADAS_NA_VRAM = [(6, 0.2, 1.0), (6, 0.4, 0.1), (7, 0.2, 1.0)]
XGB_COMBINACOES_NAO_TESTADAS_NA_VRAM = [(6, 0.2, 0.1), (6, 0.4, 1.0), (7, 0.4, 1.0)]
XGB_PARTES = [(n, lr, d, s, mcw)
              for combinacoes in (XGB_COMBINACOES_TESTADAS_NA_VRAM, XGB_COMBINACOES_NAO_TESTADAS_NA_VRAM)
              for n, lr in XGB_PARES
              for d, s, mcw in combinacoes
              if not (d == 7 and n == 150)]
XGB_LINHAS_REFIT = 670_000

MLP_ARQUITETURAS_TESTADAS_NA_VRAM = [(512,), (1024,), (2048,), (1024, 512), (2048, 1024)]
MLP_ARQUITETURAS_NAO_TESTADAS_NA_VRAM = [(3072,), (3072, 1024), (1024, 512, 256), (2048, 1024, 512)]
MLP_LAYER_NORM = [False, True]
MLP_DROPOUTS = [0.0, 0.2]
MLP_EPOCAS = [10, 20]
MLP_LRS = [3e-3, 1e-2]
MLP_LR_LINEAR = 3e-2
MLP_PARTES = ([(h, ln, d, e, lr)
               for arquiteturas in (MLP_ARQUITETURAS_TESTADAS_NA_VRAM, MLP_ARQUITETURAS_NAO_TESTADAS_NA_VRAM)
               for h in arquiteturas
               for ln in MLP_LAYER_NORM
               for d in MLP_DROPOUTS
               for e in MLP_EPOCAS
               for lr in MLP_LRS]
              + [((), False, 0.0, e, MLP_LR_LINEAR) for e in MLP_EPOCAS])

# Usado para o GridSearchCV
# MODELOS = [
#     ('RandomForestClassifier', RandomForestClassifier(n_jobs=20, random_state=0),
#      [{**RF_BASE, 'n_estimators': [T], 'max_leaf_nodes': [L]} for T, L in RF_PARES]),
#     ('XGBClassifier',
#      XGBClassifierRotulos(tree_method='hist', device='cuda', max_bin=32, sampling_method='gradient_based',
#                           colsample_bytree=0.8, n_jobs=20, random_state=0),
#      [{'n_estimators': [n], 'learning_rate': [lr], 'max_depth': [d], 'subsample': [s], 'min_child_weight': [mcw]}
#       for n, lr, d, s, mcw in XGB_PARTES],
#      {'linhas_refit': XGB_LINHAS_REFIT}),
#     ('MLPClassifier',
#      MLPClassifierTorch(batch_size=2048, device='cuda', random_state=0),
#      [{'hidden_layer_sizes': [h], 'layer_norm': [ln], 'dropout': [d], 'max_iter': [e], 'learning_rate_init': [lr]}
#       for h, ln, d, e, lr in MLP_PARTES]),
# ]

MODELOS = [
    ('RandomForestClassifier', RandomForestClassifier(n_jobs=20, random_state=0),
     grid_otimizado('RandomForestClassifier')),
    ('XGBClassifier',
     XGBClassifierRotulos(tree_method='hist', device='cuda', max_bin=32, sampling_method='gradient_based',
                          colsample_bytree=0.8, n_jobs=20, random_state=0),
     grid_otimizado('XGBClassifier'),
     {'linhas_refit': XGB_LINHAS_REFIT}),
    ('MLPClassifier', MLPClassifierTorch(batch_size=2048, device='cuda', random_state=0),
     grid_otimizado('MLPClassifier')),
]

treina(MODELOS, 'X_train')


[RandomForestClassifier] já treinado com este grid (grid_20261118_164829_RandomForestClassifier, hash 99c9f726), pulando
[XGBClassifier] já treinado com este grid (grid_20261119_194324_XGBClassifier, hash 01891e9c), pulando
[MLPClassifier] partes em artefatos/grid_20261121_161832_MLPClassifier/ (hash fe7d2e2a): 146 partes × 5 folds


/home/lkiepper/miniforge3/envs/python_dev-general/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.003, max_iter=10;, score=0.523 total time=  40.0s
[CV 2/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.003, max_iter=10;, score=0.533 total time=  39.2s
[CV 3/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.003, max_iter=10;, score=0.543 total time=  39.2s
[CV 4/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.003, max_iter=10;, score=0.535 total time=  39.0s
[CV 5/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.003, max_iter=10;, score=0.520 total time=  39.2s
[MLPClassifier 1/146] 3.3 min; melhor 0.5308


/home/lkiepper/miniforge3/envs/python_dev-general/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.01, max_iter=10;, score=0.571 total time=  39.1s
[CV 2/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.01, max_iter=10;, score=0.570 total time=  39.1s
[CV 3/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.01, max_iter=10;, score=0.577 total time=  39.1s
[CV 4/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.01, max_iter=10;, score=0.559 total time=  39.0s
[CV 5/5] END dropout=0.0, hidden_layer_sizes=(512,), layer_norm=False, learning_rate_init=0.01, max_iter=10;, score=0.554 total time=  39.1s
[MLPClassifier 2/146] 3.3 min; melhor 0.5664


/home/lkiepper/miniforge3/envs/python_dev-general/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


... (283 outputs left out)

/home/lkiepper/miniforge3/envs/python_dev-general/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=10;, score=0.479 total time=  48.2s
[CV 2/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=10;, score=0.504 total time=  48.0s
[CV 3/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=10;, score=0.519 total time=  48.0s
[CV 4/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=10;, score=0.507 total time=  48.0s
[CV 5/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=10;, score=0.493 total time=  48.4s
[MLPClassifier 145/146] 4.0 min; melhor 0.5002


/home/lkiepper/miniforge3/envs/python_dev-general/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=20;, score=0.533 total time= 1.6min
[CV 2/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=20;, score=0.539 total time= 1.6min
[CV 3/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=20;, score=0.554 total time= 1.6min
[CV 4/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=20;, score=0.543 total time= 1.6min
[CV 5/5] END dropout=0.0, hidden_layer_sizes=(), layer_norm=False, learning_rate_init=0.03, max_iter=20;, score=0.535 total time= 1.6min
[MLPClassifier 146/146] 8.0 min; melhor 0.5409
[MLPClassifier] refit 3.2 min; melhores: {'dropout': 0.2, 'hidden_layer_sizes': (2048,), 'layer_norm': False, 'learning_rate_init': 0.01, 'max_iter': 10}


In [ ]:
#@title Treina os algoritmos (Notebook pessoal)
import numpy as np
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC


SVM_C = np.geomspace(0.1, 3, 53)

# Usado no GridSearchCV
# MODELOS = [
#     ('LinearSVC', OneVsRestClassifier(LinearSVC(random_state=0), n_jobs=10),
#      [{'estimator__C': [float(c)]} for c in SVM_C]),
# ]

MODELOS = [
    ('LinearSVC', OneVsRestClassifier(LinearSVC(random_state=0), n_jobs=10),
     grid_otimizado('LinearSVC')),
]

treina(MODELOS, 'X_train')


[LinearSVC] partes em artefatos/grid_20261119_195549_LinearSVC/ (hash 9b412d73): 53 partes × 5 folds
[LinearSVC 1/53] já salva, pulando
[LinearSVC 2/53] já salva, pulando
[LinearSVC 3/53] já salva, pulando
[LinearSVC 4/53] já salva, pulando
[LinearSVC 5/53] já salva, pulando
[LinearSVC 6/53] já salva, pulando
[LinearSVC 7/53] já salva, pulando
[LinearSVC 8/53] já salva, pulando
[LinearSVC 9/53] já salva, pulando
[LinearSVC 10/53] já salva, pulando
[LinearSVC 11/53] já salva, pulando
[LinearSVC 12/53] já salva, pulando
[LinearSVC 13/53] já salva, pulando
[LinearSVC 14/53] já salva, pulando
[LinearSVC 15/53] já salva, pulando
[LinearSVC 16/53] já salva, pulando
[LinearSVC 17/53] já salva, pulando
[LinearSVC 18/53] já salva, pulando
[LinearSVC 19/53] já salva, pulando
[LinearSVC 20/53] já salva, pulando
[LinearSVC 21/53] já salva, pulando
[LinearSVC 22/53] já salva, pulando
[LinearSVC 23/53] já salva, pulando
[LinearSVC 24/53] já salva, pulando
[LinearSVC 25/53] já salva, pulando
[LinearS

/home/leo/miniforge3/envs/python_dev-general/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END ...estimator__C=1.2818610191887019;, score=0.486 total time= 5.8min
[CV 2/5] END ...estimator__C=1.2818610191887019;, score=0.496 total time= 5.2min
[CV 3/5] END ...estimator__C=1.2818610191887019;, score=0.517 total time= 5.2min
[CV 4/5] END ...estimator__C=1.2818610191887019;, score=0.526 total time= 5.2min
[CV 5/5] END ...estimator__C=1.2818610191887019;, score=0.508 total time= 5.3min
[LinearSVC 40/53] 26.7 min; melhor 0.5065


/home/leo/miniforge3/envs/python_dev-general/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END ....estimator__C=1.368507301937268;, score=0.491 total time= 5.2min
[CV 2/5] END ....estimator__C=1.368507301937268;, score=0.498 total time= 5.2min
[CV 3/5] END ....estimator__C=1.368507301937268;, score=0.521 total time= 5.3min
[CV 4/5] END ....estimator__C=1.368507301937268;, score=0.525 total time= 5.3min
[CV 5/5] END ....estimator__C=1.368507301937268;, score=0.508 total time= 5.3min
[LinearSVC 41/53] 26.2 min; melhor 0.5087


/home/leo/miniforge3/envs/python_dev-general/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


... (19 outputs left out)

/home/leo/miniforge3/envs/python_dev-general/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END ....estimator__C=2.810056659633655;, score=0.508 total time= 5.1min
[CV 2/5] END ....estimator__C=2.810056659633655;, score=0.514 total time= 4.9min
[CV 3/5] END ....estimator__C=2.810056659633655;, score=0.535 total time= 5.0min
[CV 4/5] END ....estimator__C=2.810056659633655;, score=0.535 total time= 5.0min
[CV 5/5] END ....estimator__C=2.810056659633655;, score=0.524 total time= 5.1min
[LinearSVC 52/53] 25.1 min; melhor 0.5233


/home/leo/miniforge3/envs/python_dev-general/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END ..................estimator__C=3.0;, score=0.508 total time= 5.2min
[CV 2/5] END ..................estimator__C=3.0;, score=0.515 total time= 5.0min
[CV 3/5] END ..................estimator__C=3.0;, score=0.535 total time= 5.1min
[CV 4/5] END ..................estimator__C=3.0;, score=0.537 total time= 5.0min
[CV 5/5] END ..................estimator__C=3.0;, score=0.524 total time= 5.2min
[LinearSVC 53/53] 25.5 min; melhor 0.5238
[LinearSVC] refit 6.2 min; melhores: {'estimator__C': 3.0}


In [ ]:
#@title Treina os algoritmos (PC)

In [ ]:
#@title Treina os algoritmos (DGX)



BERT_ELEVEN = dict(checkpoint='google-bert/bert-base-multilingual-cased',
                   maxlen=150, batch_size=8, lr=2e-5, epocas=3, validacao=0.1)

# Usado no GridSearchCV
# MODELOS = [
#     ('BERT', ClassificadorBERT(**BERT_ELEVEN, arq_checkpoint=_caminho('checkpoint_BERT', 'pt')),
#      {}, {'cv': None}),
# ]

MODELOS = [
    ('BERT', ClassificadorBERT(**BERT_ELEVEN, arq_checkpoint=_caminho('checkpoint_BERT', 'pt')),
     grid_otimizado('BERT')),
]

treina(MODELOS, 'X_train_texto')

In [5]:
#@title Suite de inferência
import numpy as np

def infere(modelo, X, lote=2000):
    return np.concatenate([modelo.predict(X[i:i + lote]) for i in range(0, X.shape[0], lote)])

In [ ]:
#@title Inferência: ŷ_test = A(X_test)
MODELOS_INFERIDOS = {'RandomForestClassifier': 'X_test', 'XGBClassifier': 'X_test', 'LinearSVC': 'X_test',
                     'MLPClassifier': 'X_test', 'BERT': 'X_test_texto'}
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'

for semente in sementes_split():
    sufixo = f'_splitseed{semente}'
    entradas = {}
    for nome, entrada in MODELOS_INFERIDOS.items():
        rotulo = f'{nome}{sufixo}'
        if not existe(f'modelo_{rotulo}') or existe(f'yhat_test_{rotulo}'):
            continue
        if entrada not in entradas:
            entradas[entrada] = carrega(f'{entrada}{sufixo}')
        modelo = carrega(f'modelo_{rotulo}')
        if 'device' in modelo.get_params(deep=False):
            modelo.set_params(device=DISPOSITIVO)
        salva(f'yhat_test_{rotulo}', infere(modelo, entradas[entrada]))
        print(f'[{rotulo}] ŷ_test gravado')
        del modelo
    del entradas
    
# De cada máquina, trazer:
# yhat_test_{nome}_splitseed*.joblib
# treino_{nome}_splitseed*.joblib

In [2]:
#@title Suite de avaliação
import os, sys, hashlib
import numpy as np, scipy, sklearn
import pandas as pd
from sklearn.metrics import classification_report

def _medidas(y_true, y_pred):
    return {
        'classification_report': classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }

def _assegura_sem_vazamento(nome, treino, y_true):
    vazadas = treino['indice_treino'].intersection(y_true.index)
    assert len(vazadas) == 0, f'[{nome}] {len(vazadas)} linhas avaliadas foram usadas no treino'

def _relatorio_do_modelo(nome, treino, y_true, y_pred):
    _assegura_sem_vazamento(nome, treino, y_true)
    relatorio = _medidas(y_true, y_pred)['classification_report']
    return [f'--- {nome} ---',
            f'Melhores parâmetros (CV): {treino["melhores_parametros"]}',
            f'Score médio na CV: {treino["melhor_score_cv"]:.4f}; tempo de treino: {treino["tempo_s"]:.0f} s',
            f'Acurácia {relatorio["accuracy"]:.4f} | macro-F1 {relatorio["macro avg"]["f1-score"]:.4f} '
            f'(precision {relatorio["macro avg"]["precision"]:.4f}, recall {relatorio["macro avg"]["recall"]:.4f})',
            f'Parâmetros completos: {treino["parametros"]}',
            pd.DataFrame(relatorio).T.to_string(), '']

def sha256_do_arquivo(caminho):
    with open(caminho, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

def cabecalho(agora, config, arq_x, sha_x):
    return '\n'.join([f'=== Avaliação {agora:%Y-%m-%d %H:%M:%S} ===',
                      f'Dados: {config}',
                      f'Matriz TF-IDF: {arq_x} sha256={sha_x}',
                      f'Versões: python {sys.version.split()[0]}, numpy {np.__version__}, '
                      f'scipy {scipy.__version__}, scikit-learn {sklearn.__version__}', '', ''])

def avalia(treinos, y_true, yhats):
    linhas = []
    for nome, treino in treinos.items():
        linhas += _relatorio_do_modelo(nome, treino, y_true, yhats[nome])
    return '\n'.join(linhas)

def remove_split_temporario(config):
    if config['guardou_matrizes']:
        return
    for nome in ('X_train', 'X_test', 'X_train_texto', 'X_test_texto', 'y_train', 'y_test', 'config_dados'):
        for ext in ('npz', 'joblib'):
            if os.path.exists(_caminho(nome, ext)):
                os.remove(_caminho(nome, ext))
    print('Arquivos temporários do split removidos; sementes registradas no log')


In [3]:
#@title Avalia: y_test vs ŷ_test
import datetime

MODELOS_AVALIADOS = ['RandomForestClassifier', 'XGBClassifier', 'LinearSVC', 'MLPClassifier', 'BERT']

config = carrega('config_dados')
y_test = carrega('y_test')
treinos = {nome: carrega(f'treino_{nome}') for nome in MODELOS_AVALIADOS}
yhats_test = {nome: carrega(f'yhat_test_{nome}') for nome in MODELOS_AVALIADOS}

agora = datetime.datetime.now()
ARQ_X_TFIDF = _caminho('X_tfidf', 'npz')
texto = cabecalho(agora, config, ARQ_X_TFIDF, sha256_do_arquivo(ARQ_X_TFIDF)) + avalia(treinos, y_test, yhats_test)
print(texto)
with open(_caminho(f'avaliacao_{agora:%Y%m%d_%H%M%S}', 'log'), 'w') as f:
    f.write(texto + '\n')
remove_split_temporario(config)


=== Avaliação 2026-11-24 17:35:01 ===
Dados: {'usar_amostra': False, 'n_amostra': 1117623, 'semente_amostra': None, 'test_size': 0.25, 'semente_split': 345510517, 'guardou_matrizes': True, 'nivel_ncm': 4}
Matriz TF-IDF: artefatos/X_tfidf.npz sha256=bef63211aea005880f69c96490f63c20cd61fd694f1b10723976c09453b24969
Versões: python 3.13.2, numpy 2.4.6, scipy 1.18.0, scikit-learn 1.9.1

--- RandomForestClassifier ---
Melhores parâmetros (CV): {'bootstrap': False, 'class_weight': None, 'max_features': 'sqrt', 'max_leaf_nodes': 150000, 'n_estimators': np.int64(4)}
Score médio na CV: 0.5048; tempo de treino: 85954 s
Acurácia 0.9046 | macro-F1 0.5075 (precision 0.5587, recall 0.4896)
Parâmetros completos: {'bootstrap': False, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': 150000, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_c

# Resultados

1909508899
1596260650
1363945693
1375679088
827915821
1257687752
686578392
378699650
259128988